In [1]:
import os
import numpy as np
import pandas as pd
import torch
import joblib

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, jaccard_score

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

In [2]:
data_path = "/kaggle/input/datasets/mikeelio4/multilabel-tags-merged/multilabel_tags_merged.csv"


df = pd.read_csv(data_path)



In [3]:
df = df[["Title", "Tags"]].copy()
df = df.dropna(subset=["Title", "Tags"])

df["Title"] = df["Title"].astype(str).str.strip()
df["Tags"] = df["Tags"].astype(str).str.strip()

df = df[(df["Title"] != "") & (df["Tags"] != "")]



In [4]:
df["label_list"] = df["Tags"].apply(lambda x: x.split())
df[["Title", "Tags", "label_list"]].head()  

,Title,Tags,label_list
0,# + items .append is not a function,javascript,[javascript]
1,# - how to parallel code that lock several obj...,c#,[c#]
2,# . what do and # do in this code,javascript,[javascript]
3,# .dialog is not a function error,javascript,[javascript]
4,# .dialog is not a function error after using ...,javascript,[javascript]


In [5]:
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df["label_list"])

print("Number of labels:", len(mlb.classes_))
print("y shape:", y.shape)
print("First labels:", mlb.classes_[:20])

Number of labels: 50
y shape: (1866458, 50)
First labels: ['active-directory' 'algorithm' 'amazon-ec2' 'android' 'apache' 'api'
 'architecture' 'bash' 'c#' 'c++' 'centos' 'data-structures'
 'database-design' 'debugging' 'design-patterns' 'dns' 'ftp' 'git' 'http'
 'image-processing']


In [6]:
X = df["Title"].tolist()

In [7]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42
)

print("Train:", len(X_train), y_train.shape)
print("Val  :", len(X_val), y_val.shape)
print("Test :", len(X_test), y_test.shape)

Train: 1493166 (1493166, 50)
Val  : 186646 (186646, 50)
Test : 186646 (186646, 50)


In [8]:
y_train = y_train.astype(np.float32)
y_val = y_val.astype(np.float32)
y_test = y_test.astype(np.float32)

train_size = min(650000, len(X_train))
val_size = min(50000, len(X_val))
test_size = min(50000, len(X_test))

train_idx = np.random.RandomState(42).choice(len(X_train), train_size, replace=False)
val_idx = np.random.RandomState(42).choice(len(X_val), val_size, replace=False)
test_idx = np.random.RandomState(42).choice(len(X_test), test_size, replace=False)

train_df = pd.DataFrame({
    "title": X_train.iloc[train_idx].values if hasattr(X_train, "iloc") else np.array(X_train)[train_idx],
    "labels": list(y_train[train_idx])
})

val_df = pd.DataFrame({
    "title": X_val.iloc[val_idx].values if hasattr(X_val, "iloc") else np.array(X_val)[val_idx],
    "labels": list(y_val[val_idx])
})

test_df = pd.DataFrame({
    "title": X_test.iloc[test_idx].values if hasattr(X_test, "iloc") else np.array(X_test)[test_idx],
    "labels": list(y_test[test_idx])
})
print(y_train.dtype, y_val.dtype, y_test.dtype)

float32 float32 float32


In [9]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

print(train_dataset)
print(val_dataset)
print(test_dataset)

Dataset({
    features: ['title', 'labels'],
    num_rows: 650000
})
Dataset({
    features: ['title', 'labels'],
    num_rows: 50000
})
Dataset({
    features: ['title', 'labels'],
    num_rows: 50000
})


In [10]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [11]:
def tokenize_function(example):
    return tokenizer(
        example["title"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/650000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [12]:
num_labels = y_train.shape[1]

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

print("Num labels:", num_labels)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Num labels: 50


In [13]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/distilbert_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    per_device_train_batch_size=20,
    per_device_eval_batch_size=20,
    num_train_epochs=5,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_at_3",
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

In [14]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

def top_k_binary_predictions(y_scores, k):
    y_pred = np.zeros_like(y_scores, dtype=int)
    topk_idx = np.argsort(-y_scores, axis=1)[:, :k]

    for i in range(y_scores.shape[0]):
        y_pred[i, topk_idx[i]] = 1

    return y_pred

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    results = {}

    for k in [1, 2, 3, 4,5]:
        preds_k = top_k_binary_predictions(logits, k)

        results[f"precision_at_{k}"] = precision_score(labels, preds_k, average="micro", zero_division=0)
        results[f"recall_at_{k}"] = recall_score(labels, preds_k, average="micro", zero_division=0)
        results[f"f1_at_{k}"] = f1_score(labels, preds_k, average="micro", zero_division=0)

    return results

In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [16]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Precision At 1,Recall At 1,F1 At 1,Precision At 2,Recall At 2,F1 At 2,Precision At 3,Recall At 3,F1 At 3,Precision At 4,Recall At 4,F1 At 4,Precision At 5,Recall At 5,F1 At 5
1,0.125192,0.070427,0.764240,0.671316,0.714771,0.455660,0.800513,0.580751,0.323967,0.853727,0.469696,0.252015,0.885490,0.392362,0.206568,0.907257,0.336517
2,0.067726,0.064182,0.784540,0.689148,0.733757,0.467670,0.821612,0.596058,0.330893,0.871980,0.479739,0.256730,0.902057,0.399703,0.209996,0.922313,0.342101
3,0.062371,0.062079,0.790900,0.694735,0.739705,0.472330,0.829799,0.601997,0.333860,0.879798,0.484040,0.258620,0.908698,0.402645,0.211104,0.927180,0.343906
4,0.059478,0.061132,0.793940,0.697405,0.742548,0.474040,0.832803,0.604177,0.335160,0.883224,0.485925,0.259355,0.911281,0.403789,0.211832,0.930377,0.345092
5,0.057763,0.060858,0.795480,0.698758,0.743989,0.474630,0.833840,0.604929,0.335473,0.884050,0.486379,0.259430,0.911544,0.403906,0.211988,0.931062,0.345346


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=13545, training_loss=0.07450595206552521, metrics={'train_runtime': 9188.4852, 'train_samples_per_second': 353.704, 'train_steps_per_second': 1.474, 'total_flos': 5.386094688e+16, 'train_loss': 0.07450595206552521, 'epoch': 5.0})

In [17]:
val_results = trainer.evaluate(eval_dataset=val_dataset)
print("Validation Results:", val_results)

test_results = trainer.evaluate(eval_dataset=test_dataset)
print("Test Results:", test_results)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Validation Results: {'eval_loss': 0.060857877135276794, 'eval_precision_at_1': 0.79548, 'eval_recall_at_1': 0.6987579276541172, 'eval_f1_at_1': 0.7439885522956201, 'eval_precision_at_2': 0.47463, 'eval_recall_at_2': 0.8338398833470951, 'eval_f1_at_2': 0.6049285946431644, 'eval_precision_at_3': 0.33547333333333335, 'eval_recall_at_3': 0.8840498234395039, 'eval_f1_at_3': 0.48637885956476146, 'eval_precision_at_4': 0.25943, 'eval_recall_at_4': 0.9115440698511973, 'eval_f1_at_4': 0.403906259122454, 'eval_precision_at_5': 0.211988, 'eval_recall_at_5': 0.9310623495722141, 'eval_f1_at_5': 0.3453461965782726, 'eval_runtime': 54.7588, 'eval_samples_per_second': 913.096, 'eval_steps_per_second': 3.817, 'epoch': 5.0}


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Test Results: {'eval_loss': 0.06134815514087677, 'eval_precision_at_1': 0.79192, 'eval_recall_at_1': 0.6947154186258685, 'eval_f1_at_1': 0.7401398183109649, 'eval_precision_at_2': 0.47397, 'eval_recall_at_2': 0.8315846726086041, 'eval_f1_at_2': 0.6037988228999465, 'eval_precision_at_3': 0.3351866666666667, 'eval_recall_at_3': 0.8821320794441715, 'eval_f1_at_3': 0.4857871649693714, 'eval_precision_at_4': 0.259605, 'eval_recall_at_4': 0.9109586637658783, 'eval_f1_at_4': 0.4040607635916512, 'eval_precision_at_5': 0.212032, 'eval_recall_at_5': 0.9300301775563198, 'eval_f1_at_5': 0.34533348968716204, 'eval_runtime': 54.3923, 'eval_samples_per_second': 919.248, 'eval_steps_per_second': 3.842, 'epoch': 5.0}
